<div style="background:#102A43;color:#ffffff;border-radius:11px;padding:30px 28px 24px;border-top:7px solid #18B7AC;">
<div style="color:#67DED3;letter-spacing:2px;font-weight:700;font-size:12px;">INTRODUCTION TO MACHINE LEARNING · GAZA SKY GEEKS</div>
<div style="font-size:32px;font-weight:750;line-height:1.25;margin:12px 0 9px;">Lecture 08 · Feature Engineering:<br>Transforming Data for Learning</div>
<div style="font-size:15px;color:#DDEAF3;">Instructor: Diaeddin Rimawi &nbsp; | &nbsp; 10 November 2026 &nbsp; | &nbsp; Part 2: Data Preprocessing and Proximity Measures</div>
<div style="font-size:17px;color:#ffffff;border-top:1px solid #466178;padding-top:16px;margin-top:18px;"><strong>Today's challenge:</strong> We can recognize problems in our observations. How do we represent valid measurements and categories so that Machine Learning algorithms can use them meaningfully?</div>
</div>

We return to **Palmer Penguins**, our familiar dataset. Our hypothetical future goal is to identify penguin species. **We will not train a prediction model today.**

### What you will be able to do

By the end of this lecture, you should be able to:

- Distinguish a **feature** from a **prediction target**, and data cleaning from feature engineering.
- Construct and interpret a useful feature from existing physical measurements.
- Explain what **log transforms** and **discretization** change—and what information may disappear.
- Explain and calculate **min–max scaling** and **standardization**, including when scale matters.
- Represent **nominal categories** numerically without inventing an order.
- Design and audit a small feature table while keeping `species` separate.
- Explain why learned transformation parameters belong to the **training data**, not the validation or test data.

**Guiding principle:** More columns are not necessarily better. A feature needs a meaning, a purpose, and a justified representation.

### From L07 to L08: Is clean data enough?

In **L07**, we asked whether data were valid, complete, and consistent. We looked at missing values, duplicates, suspicious values, and decisions that might accidentally introduce leakage.

Now imagine we have investigated those problems and are satisfied with the observations we can use.

- Body mass is measured in **grams**, while flipper length is in **millimeters**.
- Island is represented by **words**, not numbers.
- Two bill measurements may together express a characteristic of **shape**.
- A numerical value may be detailed, but a simplified category may sometimes be easier to interpret.

**Discussion:** A clean dataset contains reliable information. Does it follow that every attribute is already represented in the most useful way for a learning algorithm?

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">01 · PREPARING OUR INVESTIGATION</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">One dataset, different representations</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">First identify the data and its meaning; do not automatically transform every column.</div>
</div>

### Use the same local scientific observations

We work with the **344 original penguin observations** and eight recorded attributes. Our hypothetical question is: *Can physical measurements and suitable context help identify a penguin's species?*

- **Possible target:** `species` (the answer we would eventually like to predict).
- **Candidate measurements:** bill length, bill depth, flipper length, and body mass.
- **Possible context:** `island` and `sex`; usefulness depends on the intended deployment.
- **Questionable without explanation:** `year`—a number may represent a date rather than a continuous biological characteristic.

We keep the original dataset untouched. We will explicitly select complete **physical measurements** for demonstrations; this is not a universal cleaning rule.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SPECIES_ORDER = ["Adelie", "Chinstrap", "Gentoo"]
SPECIES_COLORS = {"Adelie": "#158B94", "Chinstrap": "#DB9137", "Gentoo": "#557AB5"}
TEAL, NAVY = "#158B94", "#102A43"

# The notebook and data/ folder are distributed together.
data_path = Path("data/penguins.csv")
penguins = pd.read_csv(data_path)
assert penguins.shape == (344, 8), "Unexpected source file: check data/penguins.csv"
print("Original observations and attributes:", penguins.shape)
display(penguins.head(4))

### Two data-quality decisions, made visible

- Four physical measurements are needed to build the examples. **Two rows** lack one or more of them; we exclude those rows *only from the numerical working table*.
- `sex` has more missing values. We will **not** silently remove those rows, because `sex` is not required for our chosen feature table.
- We are not declaring the excluded observations incorrect; they simply cannot support the examples without additional missing-value decisions.

We also attach **illustrative train/validation/test roles** to our working copy. The roles help us avoid fitting transformation parameters on evaluation observations. 

In [ ]:
measurement_columns = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
study = penguins.dropna(subset=measurement_columns).copy()

rng = np.random.default_rng(8)
row_order = rng.permutation(study.index.to_numpy())
n_train = int(0.70 * len(study))
n_val = int(0.20 * len(study))
study["split_role"] = "test"
study.loc[row_order[:n_train], "split_role"] = "train"
study.loc[row_order[n_train:n_train+n_val], "split_role"] = "validation"

print("Original rows:", len(penguins), "| Rows with all four measurements:", len(study))
print("Missing sex in working table:", study["sex"].isna().sum())
print("Illustrative split roles:")
display(study["split_role"].value_counts().reindex(["train", "validation", "test"]))

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">02 · WHAT COUNTS AS A FEATURE?</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Inputs, targets, and useful representations</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">A feature is defined by the question we want to answer—not only by its data type.</div>
</div>

### Feature, target, and feature engineering

- A **feature** is an attribute supplied as *input* to a learning algorithm.
  - Example: a penguin's flipper length.
- A **target** is the answer a supervised model would aim to predict.
  - For species identification, our target would be `species`.
- **Feature engineering** means constructing or changing the representation of information so that a learning algorithm can use relevant patterns.
  - The information may remain the same even when the numbers or columns change.

**Important distinction:** Choosing *which* of many candidate features to keep is **feature selection**; we study formal selection methods in **L10**. Today we focus on how to **represent** the candidate information.

### One observation can be represented in several ways

| Recorded information | Possible representation | What is the reason? |
|:--|:--|:--|
| Body mass (g) | Body mass (kg) | A more readable unit; no new biological evidence. |
| Bill length and depth | Bill length ÷ bill depth | A dimensionless description of bill proportions. |
| Flipper length (mm) | Standardized length | Express differences relative to a reference distribution. |
| Island name | Island indicator columns | Represent nominal categories numerically. |
| Flipper length (mm) | Short / medium / long interval | Simplify a continuous measurement, sacrificing detail. |

**Question:** Which transformations preserve enough information to reconstruct the original values? Which ones might erase distinctions between penguins?

### What information would we actually have when predicting?

Suppose researchers measure a new penguin and want to identify its species.

- Physical measurements are plausible inputs **if they can be taken before identifying the species**.
- An island label can be available, but it may be a **shortcut** tied to where the original observations were collected. A model that relies on island could generalize poorly to new locations.
- Sex may not always be known; a missing category cannot automatically be interpreted as one of the known sexes.
- `species` is the **target**, not a feature. Using the true species to create a prediction-time input would be **target leakage**.

A feature can be informative in one task but inappropriate or unavailable in another. **Availability, meaning, and future use matter.**

In [ ]:
candidate_inputs = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "island"]
raw_candidate_features = study[candidate_inputs].copy()
y_species = study["species"].copy()   # Keep the hypothetical target separate.
print("Candidate input columns:", raw_candidate_features.columns.tolist())
print("Target name:", y_species.name)
display(raw_candidate_features.head(3))

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">03 · CONSTRUCTING FEATURES</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Can two measurements reveal a new relationship?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Start with interpretable combinations, not a large collection of arbitrary formulas.</div>
</div>

### First: a unit conversion is not new information

Body mass is recorded in grams. For humans, **4.2 kg** may be easier to read than **4,200 g**.

- Divide by 1,000 to convert grams to kilograms.
- This changes **units and numerical magnitude**, but not the penguin's mass.
- We can recover the original value by multiplying by 1,000.
- Therefore, this representation is **reversible** and adds **no independent biological evidence**.

Why might the smaller number matter to an algorithm that processes numerical differences? We will return to this puzzle when we discuss scaling.

In [ ]:
study["body_mass_kg"] = study["body_mass_g"] / 1000

display(study[["body_mass_g", "body_mass_kg"]].head())
print("Back-conversion matches the original grams:",
      np.allclose(study["body_mass_kg"] * 1000, study["body_mass_g"]))

### Can a ratio describe shape rather than absolute size?

A bill is measured in two directions. Instead of treating these lengths independently, we can express their **proportion**:

$$
\text{Bill shape ratio} = \frac{\text{Bill length (mm)}}{\text{Bill depth (mm)}}.
$$

- A larger ratio describes a **longer bill relative to its depth**.
- Millimeters cancel, so the ratio has **no physical unit**.
- Different bills can have the same ratio (for example, $40/20 = 50/25 = 2$). The ratio **does not preserve both original measurements**.
- The denominator must be meaningful and nonzero. We checked that the required recorded measurements are present; in another dataset we would also validate the denominator.

**Hypothesis:** Bill proportions may differ between species. A hypothesis is not yet a demonstration of predictive improvement.

In [ ]:
assert (study["bill_depth_mm"] > 0).all(), "A zero/nonpositive denominator needs review"
study["bill_shape_ratio"] = study["bill_length_mm"] / study["bill_depth_mm"]

display(study[["species", "bill_length_mm", "bill_depth_mm", "bill_shape_ratio"]].head())
ratio_summary = study.groupby("species")["bill_shape_ratio"].agg(["count", "mean", "median", "std"])
display(ratio_summary.reindex(SPECIES_ORDER).round(2))

### Is the relationship visible in real observations?

We can compare the **original bill lengths** with the **constructed shape ratios**, separately for the three known species. We use species labels **only to interpret the exploration**, not to construct the ratio.

**Before running:** Predict whether the species with the longest bills must also have the largest ratio. Could a deep bill change that ranking?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.4, 4), constrained_layout=True)
for ax, column, ylabel in [
    (axes[0], "bill_length_mm", "Bill length (mm)"),
    (axes[1], "bill_shape_ratio", "Bill length / bill depth (unitless)"),
]:
    measurements = [study.loc[study["species"] == s, column] for s in SPECIES_ORDER]
    boxes = ax.boxplot(measurements, tick_labels=SPECIES_ORDER, patch_artist=True,
                       medianprops={"color": NAVY, "linewidth": 2},
                       flierprops={"marker": ".", "markersize": 4, "alpha": 0.6})
    for patch, species in zip(boxes["boxes"], SPECIES_ORDER):
        patch.set_facecolor(SPECIES_COLORS[species]); patch.set_alpha(0.75)
    ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=0.2)
axes[0].set_title("Original measurement: length")
axes[1].set_title("Constructed feature: proportions")
plt.show()

### What did we gain—and what did we lose?

- **Observation:** The ratio displays a different aspect of the measurements. On average, the **Gentoo** group has the largest bill-shape ratio in our dataset, even though the **Chinstrap** group has the largest mean bill length.
- **Interpretation:** The ratio emphasizes **relative proportions**, not simply size.
- **Limitation:** Two different original pairs may become the same ratio. The feature can also reflect measurement noise in either component.
- **Scientific caution:** Clearer group differences in a plot **do not prove** a future model will make better predictions. That requires model training and evaluation in later lectures.


<div style="border-left:5px solid #18B7AC;background:#EEF8F7;padding:12px 16px;border-radius:5px;"><strong style="color:#102A43;">ACTIVITY 1 — A different ratio, a different story?</strong></div>

**Your challenge:** We used **bill length ÷ bill depth**. Investigate a *different* relationship: **flipper length per kilogram of body mass**.

1. Construct `flipper_mm_per_kg` using the working table `study`. Work on a **copy**.
2. Compare the **mean flipper length** and the **mean new ratio** across species.
3. Does the species with the longest average flippers also have the largest average ratio?

**Discovery to look for:** A derived feature can change the ordering of groups even though we have not collected any new observations.

**Explain:** What is the unit of your new feature, and why is this ratio not guaranteed to be a better predictor?

In [ ]:
# Activity 1 — Work on a copy; do not change study.
activity1 = study.copy()
# TODO: construct flipper_mm_per_kg using flipper_length_mm and body_mass_kg.
# TODO: group by species and compare both average measures.
# TODO: explain the difference in group rankings.

<details>
<summary><strong>Suggested answer — click to expand after trying</strong></summary>

**Suggested code**

```python
activity1 = study.copy()
activity1["flipper_mm_per_kg"] = (
    activity1["flipper_length_mm"] / activity1["body_mass_kg"]
)
comparison = activity1.groupby("species")[[
    "flipper_length_mm", "flipper_mm_per_kg"
]].mean().reindex(SPECIES_ORDER)
print(comparison.round(2))
print("Highest mean flipper length:", comparison["flipper_length_mm"].idxmax())
print("Highest mean ratio:", comparison["flipper_mm_per_kg"].idxmax())
```

- **Expected discovery:** Gentoo has the largest average flipper length, but **Chinstrap** has the largest average flipper length per kilogram of body mass in this sample (slightly above Adelie).
- **Unit:** mm/kg. This is a relative-size description, not a new measurement and not a standard biological law.
- **Decision:** The ratio might answer a useful question, but it combines two noisy quantities and can discard absolute size. Keep the original measurements until there is evidence that the representation is useful.

**Bridge:** Not every transformation combines two measurements. What happens if we transform a *single* numerical measurement?

</details>

### Other combinations: possible does not mean useful

- **Difference:** Bill length minus bill depth (mm) emphasizes an absolute gap.
- **Sum or product:** May be meaningful when grounded in a real scientific quantity, but arbitrary arithmetic can be misleading.
- **Ratio:** Can remove a common scale and highlight proportions, but may be unstable with small denominators.

**Decision rule:** Before creating a feature, complete this sentence: *“This new value represents ___, and it might matter because ___.”* If we cannot explain it, we should reconsider creating it.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">04 · TRANSFORMING CONTINUOUS MEASUREMENTS</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Should numerical measurements always stay in their original form?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">A transformation can make patterns easier to express—or hide distinctions we care about.</div>
</div>

### A skewed distribution: why consider a logarithm?

Imagine a **separate, illustrative dataset** measuring the amount of data transferred by nine computer sessions (in MB):

`1, 2, 3, 4, 6, 10, 30, 120, 900`.

- Most sessions are small, but one is much larger. The distribution is **strongly right-skewed**.
- A **logarithm** compresses large positive values more than small ones.
- The new scale may reveal relative differences that are hard to see on the original scale.
- The numbers now describe **logarithmic values**; differences must be interpreted differently.

We do **not** need this transformation for every penguin measurement. This example is deliberately constructed because penguin lengths do not have such an extreme range.

In [ ]:
# Illustrative session sizes, not penguin observations.
session_mb = np.array([1, 2, 3, 4, 6, 10, 30, 120, 900], dtype=float)
log_session_mb = np.log10(session_mb)   # All values are strictly positive.

fig, axes = plt.subplots(1, 2, figsize=(10.8, 3.7), constrained_layout=True)
# axes[0].hist(session_mb, bins=[0, 10, 50, 150, 450, 950], color="#86BDC2", edgecolor="white")
axes[0].hist(session_mb, bins=6, color="#86BDC2", edgecolor="white")
axes[0].set(title="Original: very different session sizes", xlabel="Data transferred (MB)", ylabel="Number of sessions")
axes[1].hist(log_session_mb, bins=6, color=TEAL, edgecolor="white")
axes[1].set(title="After log10: large values compressed", xlabel="log10(data transferred in MB)", ylabel="Number of sessions")
for ax in axes: ax.grid(axis="y", alpha=0.18)
plt.show()
print("Original 10 vs 1000 differs by a factor of 100; in log10 the gap is 2.")

### Read a logarithmic value correctly

For positive $x$, using base 10:

$$
\log_{10}(1)=0,\qquad \log_{10}(10)=1,\qquad \log_{10}(100)=2.
$$

- A **tenfold multiplication** of the original value corresponds to **adding one** after the log transform.
- Logarithms are not directly defined for zero or negative real measurements. A common alternative, `np.log1p(x)`, computes $\ln(1+x)$ for nonnegative values; it is **not** the same scale as `log10(x)`.
- A logarithm is monotonic for positive values: larger measurements remain larger.
- Neither a log transform nor standardization **guarantees a normal distribution**.

**Tradeoff:** We preserve ordering, but change distances and interpretation on the numerical scale.

### Discretization: can we replace precise numbers with categories?

Consider flipper length. Instead of retaining an exact number, we might report an **interval**:

| Flipper length | Label |
|:--|:--|
| 170–190 mm (including 190) | Shorter |
| Over 190–210 mm | Middle |
| Over 210–240 mm | Longer |

- **Discretization** (binning) maps a range of continuous values to a smaller number of groups.
- The labels are **not established biological standards**; we choose the boundaries for illustration.
- The new representation is easier to summarize, but **different values in one bin become indistinguishable**.
- Boundaries matter: 190 and 191 mm fall into different categories even though they are close.

Does every learning task benefit from erasing this detail? Certainly not.

In [ ]:
# These are fixed teaching boundaries, not boundaries estimated from the dataset.
flipper_edges = [170, 190, 210, 240]
flipper_labels = ["Shorter", "Middle", "Longer"]
study["flipper_group"] = pd.cut(
    study["flipper_length_mm"], bins=flipper_edges,
    labels=flipper_labels, include_lowest=True
)
assert study["flipper_group"].notna().all(), "Check whether values fall outside chosen bins"
print("Number of observations per illustrative interval:")
display(study["flipper_group"].value_counts(sort=False))
display(study[["flipper_length_mm", "flipper_group"]].head(8))

In [ ]:
fig, ax = plt.subplots(figsize=(9.8, 3.8))
ax.hist(study["flipper_length_mm"], bins=np.arange(170, 241, 4),
        color="#86BDC2", edgecolor="white")
for boundary in [190, 210]:
    ax.axvline(boundary, color="#B46A33", linestyle="--", linewidth=1.8,
               label=f"Bin boundary: {boundary} mm")
ax.set(title="Flipper lengths: continuous variation vs chosen boundaries",
       xlabel="Flipper length (mm)", ylabel="Number of penguins")
ax.legend(frameon=False); ax.grid(axis="y", alpha=0.2)
plt.show()

### Two common ways to choose bins

| Approach | What sets the boundaries? | Possible concern |
|:--|:--|:--|
| **Fixed edges** (`pd.cut`) | Meaningful external thresholds or a chosen set of intervals. | Uneven numbers of observations; boundaries may be arbitrary. |
| **Quantile-based** (`pd.qcut`) | Observed data quantiles, aiming for similar counts. | Boundaries depend on the fitted data and may change with a new sample. |

Both methods can lose numerical detail. If boundaries are **estimated from data**, they should be determined on **training observations** and reused on validation/test data.


<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">05 · WHY SCALE MATTERS</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Can the choice of measurement unit change a comparison?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">A large number is not the same thing as an informative feature.</div>
</div>

### Mass in grams and length in millimeters

Our measurements have different numerical magnitudes:

- Body mass: usually **thousands of grams**.
- Flipper length: usually **hundreds of millimeters**.
- If a method compares raw numerical differences, mass may dominate **just because of its measurement unit**.
- This does **not** establish that mass is more predictive than flipper length.

Some methods are sensitive to feature scales; others are relatively insensitive. Today we study the representation problem, leaving formal **distance measures** to L09.

**A puzzle:** A penguin does not change when we express its mass in kilograms. Why might a numerical comparison change dramatically?

In [ ]:
scale_summary = study[measurement_columns].agg(["min", "max", "mean", "std"]).T
scale_summary["range"] = scale_summary["max"] - scale_summary["min"]
display(scale_summary[["min", "max", "range", "mean"]].round(1))
print("Mass range in grams:", scale_summary.loc["body_mass_g", "range"])
print("Flipper range in millimeters:", scale_summary.loc["flipper_length_mm", "range"])

### Min–max scaling: make the reference minimum 0 and maximum 1

For a nonconstant measurement in a chosen reference dataset:

$$
x_{\text{scaled}} = \frac{x-x_{\min}}{x_{\max}-x_{\min}}.
$$

**Small worked example (illustrative values):** Suppose the reference training masses are **3,000, 4,000, and 5,000 g**.

- $3,000 \rightarrow (3,000-3,000)/2,000=0$.
- $4,000 \rightarrow (4,000-3,000)/2,000=0.5$.
- $5,000 \rightarrow (5,000-3,000)/2,000=1$.

The transformation is linear: within the **reference range**, relative numerical differences are preserved. We must record the **reference minimum and maximum** to transform future observations in the same way.

In [ ]:
reference_mass = pd.Series([3000, 4000, 5000], name="Training mass (g)")
ref_min, ref_max = reference_mass.min(), reference_mass.max()
scaled_example = (reference_mass - ref_min) / (ref_max - ref_min)
display(pd.DataFrame({"Mass (g)": reference_mass, "Scaled": scaled_example}))

new_mass_g = 5500       # Illustrative new observation, outside the training range
print("New 5,500 g value on training scale:", (new_mass_g - ref_min) / (ref_max - ref_min))

### What does min–max scaling assume?

- The **training minimum** becomes 0 and the **training maximum** becomes 1.
- A validation or test value outside the training range can be **below 0 or above 1**. It should not be silently clipped unless that is a justified separate decision.
- An extreme value may stretch the range, squeezing most observations close together.
- A constant feature has a zero denominator; it needs explicit handling rather than blindly applying the formula.
- Scaling does **not** remove outliers or guarantee an improved model.

**Question:** Could we describe a value relative to a *typical value and a typical amount of variation* instead of the minimum and maximum?

### Standardization: describe a value relative to the mean

A **z-score** (standardized value) is:

$$
z = \frac{x-\mu}{\sigma}.
$$

- Subtract the **reference mean**: the reference average becomes **zero**.
- Divide by the **reference standard deviation**: values become differences measured in **standard-deviation units**.
- A positive z-score is **above** the reference mean; a negative score is **below** it.
- A score of $+2$ is two reference standard deviations above the mean, not “twice as heavy.”

**Convention:** We use **Pandas `.std()`**, which defaults to **sample standard deviation (`ddof=1`)**. With the same convention used to fit and check it, the reference training sample has mean 0 and sample standard deviation 1. For a constant feature, we would need a separate rule.

In [ ]:
train_mask = study["split_role"] == "train"
train_mass = study.loc[train_mask, "body_mass_g"]
train_mean_g = train_mass.mean()
train_sd_g = train_mass.std(ddof=1)      # Pandas sample standard deviation
assert train_sd_g > 0

study["body_mass_z"] = (study["body_mass_g"] - train_mean_g) / train_sd_g
print(f"Training mass mean: {train_mean_g:.1f} g")
print(f"Training mass sample SD: {train_sd_g:.1f} g")
print("Training z-score mean:", round(study.loc[train_mask, "body_mass_z"].mean(), 10))
print("Training z-score sample SD:", round(study.loc[train_mask, "body_mass_z"].std(ddof=1), 10))
display(study[["split_role", "body_mass_g", "body_mass_z"]].head())

### What changes when we standardize?

- A standardized mass no longer has **grams** as its unit. It measures position relative to the training reference distribution.
- Standardization is a linear transformation: it changes location and scale but **not the shape** of a distribution into a normal bell curve.
- A future observation must use the **same training mean and standard deviation**.
- Unusual values remain unusual; a large outlier can still affect the training mean and standard deviation.

Let us inspect the same two penguin features **before and after** a shared scaling rule for each column.

In [ ]:
# Fit BOTH scaling methods on the training portion; transform all rows consistently.
scaled_view = study[["split_role", "body_mass_g", "flipper_length_mm"]].copy()
for col in ["body_mass_g", "flipper_length_mm"]:
    reference = scaled_view.loc[scaled_view["split_role"] == "train", col]
    min_v, max_v = reference.min(), reference.max()
    mean_v, sd_v = reference.mean(), reference.std(ddof=1)
    assert max_v > min_v and sd_v > 0
    scaled_view[col + "_minmax"] = (scaled_view[col] - min_v) / (max_v - min_v)
    scaled_view[col + "_z"] = (scaled_view[col] - mean_v) / sd_v

comparison_columns = ["body_mass_g", "flipper_length_mm"]
print("Training-data summaries on each representation:")
summary = scaled_view.loc[scaled_view["split_role"] == "train"].drop(columns="split_role")
display(summary.agg(["min", "max", "mean", "std"]).round(2))

In [ ]:
# Visualize the two variables on the reference training rows only.
fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), constrained_layout=True)
train_scaled = scaled_view.loc[scaled_view["split_role"] == "train"]
representations = [
    (["body_mass_g", "flipper_length_mm"], "Original units", "g or mm"),
    (["body_mass_g_minmax", "flipper_length_mm_minmax"], "Min–max scaled", "Scaled value"),
    (["body_mass_g_z", "flipper_length_mm_z"], "Standardized", "Z-score"),
]
for ax, (columns, title, ylabel) in zip(axes, representations):
    values = train_scaled[columns]
    ax.boxplot([values.iloc[:, 0], values.iloc[:, 1]], tick_labels=["Mass", "Flipper"],
               boxprops={"color": NAVY}, medianprops={"color": TEAL, "linewidth": 2})
    ax.set(title=title, ylabel=ylabel)
    ax.grid(axis="y", alpha=0.2)
plt.show()

### Choosing between min–max scaling and standardization

| Question | Min–max scaling | Standardization |
|:--|:--|:--|
| **Reference landmarks** | Training minimum and maximum | Training mean and standard deviation |
| **Interpretation** | Relative position in training range | Number of training SDs from the mean |
| **Training range** | `[0, 1]` for a nonconstant feature | Not restricted to fixed bounds |
| **Unseen extreme values** | May lie outside `[0, 1]` | May have large positive/negative z-scores |
| **Sensitive to** | Extreme minima/maxima | Extreme values affecting mean and SD |

**Neither is universally better.** The right choice depends on the measurements and how a later algorithm uses numerical differences.

In **L09**, we will investigate how different representations affect **which penguins appear similar**.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">06 · CATEGORICAL FEATURES</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">How can an algorithm use an island name?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Categories have meaning, but arbitrary numerical codes can invent relationships.</div>
</div>

### Nominal categories and ordinal categories are not the same

- **Nominal:** Labels have **no intrinsic ranking**.
  - `island`: Biscoe, Dream, and Torgersen have names, not a natural order.
  - Assigning numbers `0, 1, 2` is convenient but could mislead a method that treats differences numerically.
- **Ordinal:** There is a genuine order, although numerical gaps may not be meaningful.
  - Example: `low`, `medium`, `high` risk levels; the order is meaningful but not necessarily equally spaced.

A numerical code is not universally wrong. Some algorithms treat category IDs as labels. But for algorithms that interpret the numbers as measurements, arbitrary codes are a poor representation of nominal categories.

**Question:** Is Biscoe “twice” Dream, or closer to Dream than Torgersen? The island names do not support that idea.

In [ ]:
island_codes = {"Biscoe": 0, "Dream": 1, "Torgersen": 2}
tiny_islands = pd.Series(["Biscoe", "Dream", "Torgersen", "Biscoe"], name="island")
coded_islands = tiny_islands.map(island_codes)
display(pd.DataFrame({"Island": tiny_islands, "Arbitrary integer": coded_islands}))
print("Numeric gap between the codes Biscoe and Torgersen:",
      island_codes["Torgersen"] - island_codes["Biscoe"])
print("Does a gap of 2 have a geographic or biological meaning? No.")

### One-hot encoding: separate indicators instead of invented ranks

**One-hot encoding** creates one membership column for each known nominal category.

| Island | `island_Biscoe` | `island_Dream` | `island_Torgersen` |
|:--|--:|--:|--:|
| Biscoe | 1 | 0 | 0 |
| Dream | 0 | 1 | 0 |
| Torgersen | 0 | 0 | 1 |

- A `1` indicates that an observation **belongs to the category**; a `0` indicates that it does not.
- The indicators represent **membership**, not rank.
- One original column becomes **multiple input columns**.
- The list of known categories must be applied consistently to future observations.

In Pandas, `pd.get_dummies(..., dtype=int)` gives us readable 0/1 columns.

In [ ]:
island_indicators = pd.get_dummies(tiny_islands, prefix="island", dtype=int)
display(pd.concat([tiny_islands, island_indicators], axis=1))
print("Indicator names:", island_indicators.columns.tolist())

### What happens to missing or new categories?

- **Missing category:** We do not automatically know which category it should be. Filling it with the most frequent label is a **decision**, not a fact.
- **New category:** At prediction time, we may encounter an island that was absent during training. An encoder must have an explicit policy for it.
- **Consistent columns:** If we run `pd.get_dummies()` independently on training and test data, the resulting columns may differ. A later model expects the **same feature definitions** in the **same order**.

We will build a small consistent indicator table later by fixing the categories from the training subset.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">07 · ONE COHERENT INVESTIGATION</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">Build a feature table for future species identification</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Make every choice visible; check what the final table can and cannot represent.</div>
</div>

### Our proposed representation—one possible choice

We will keep the **same 342 observations** that have all four required physical measurements.

1. **Target:** `species`, stored separately; never included among predictive input features.
2. **Candidate inputs:** bill length, bill depth, flipper length, mass, and island.
3. **Derived input:** bill-shape ratio; it describes proportions but is partly redundant with its source measurements.
4. **Scale:** standardize the five numerical inputs using **training** means and sample SDs.
5. **Nominal encoding:** use one-hot island indicators, with category definitions fixed from **training** observations.
6. **Excluded for this first design:** `sex` (missingness/availability decision), `year` (unclear role), and the teaching-only flipper bins (information loss).

These choices are **defensible starting assumptions**, not evidence of the best predictive representation.

### Step 1 — Separate the raw candidate table from the target

We deliberately compare **raw** and **engineered** representations of the **same rows**. Our `study` table contains extra exploratory columns; those must not be silently passed to a future model.

The input table is something we **construct explicitly**, not simply “every column except the last one.”

In [ ]:
raw_features = study[[
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "island"
]].copy()
species_target = study["species"].copy()

print("Raw candidate features:", raw_features.shape)
print("Separate species target:", species_target.shape)
assert "species" not in raw_features.columns
display(raw_features.head(3))

### Step 2 — Apply data-dependent transformations consistently

- We already defined `bill_shape_ratio` from physical measurements using a **fixed formula**; it needs no learned parameters.
- We will estimate each numerical feature's **mean and sample SD using training rows only**.
- All rows—training, validation, and test—use those **same reference statistics**.
- We use island category names observed in **training only**, and apply that fixed list to all rows.
- In this dataset, all three observed island labels appear in the training subset. If an unseen island occurred later, we would need an explicitly documented unknown-category policy; all-zero indicators can be ambiguous without one.

This code demonstrates the idea, **not** a reusable production pipeline. L10 will introduce tools to automate the fit/transform sequence safely.

In [ ]:
numeric_inputs = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm",
    "body_mass_g", "bill_shape_ratio"
]
training_rows = study["split_role"] == "train"
engineered_features = pd.DataFrame(index=study.index)

# Numeric parameters learned only from the training rows.
training_statistics = {}
for column in numeric_inputs:
    training_values = study.loc[training_rows, column]
    center = training_values.mean()
    spread = training_values.std(ddof=1)
    assert spread > 0, f"Cannot standardize a constant feature: {column}"
    training_statistics[column] = (center, spread)
    engineered_features[column + "_z"] = (study[column] - center) / spread

# A fixed training vocabulary prevents different subsets from making different columns.
training_islands = sorted(study.loc[training_rows, "island"].dropna().unique().tolist())
assert set(study["island"].dropna().unique()) <= set(training_islands), (
    "An unseen island needs an explicit unknown-category policy"
)
island_categories = pd.Categorical(study["island"], categories=training_islands)
island_onehot = pd.get_dummies(
    pd.Series(island_categories, index=study.index, name="island"),
    prefix="island", dtype=int
)
engineered_features = pd.concat([engineered_features, island_onehot], axis=1)

assert engineered_features.index.equals(species_target.index)
assert "species" not in engineered_features.columns
assert engineered_features.isna().sum().sum() == 0
print("Training island vocabulary:", training_islands)
print("Engineered input shape:", engineered_features.shape)
display(engineered_features.head(4).round(2))

### Step 3 — Audit the result instead of trusting its appearance

We need to inspect:

- **Rows:** Did we keep the observations we intended to keep? Which rows were excluded?
- **Columns:** Did one island column become multiple indicators? Did we add useful information—or just more columns?
- **Values:** Are there missing results, unexpected ranges, or non-numerical types?
- **Roles:** Is `species` absent from `engineered_features`? Does `split_role` remain **metadata**, not an input?
- **Recoverability:** Standardized measurements can be recovered using saved training parameters; bin labels alone cannot recover the original flipper lengths.

We are creating a candidate representation, not claiming it improves prediction.

In [ ]:
audit = pd.DataFrame({
    "Representation": ["Original CSV", "Rows with required measurements", "Raw candidate inputs", "Engineered inputs"],
    "Rows": [len(penguins), len(study), len(raw_features), len(engineered_features)],
    "Columns": [penguins.shape[1], study.shape[1], raw_features.shape[1], engineered_features.shape[1]],
})
display(audit)
print("Raw columns:", raw_features.columns.tolist())
print("Engineered columns:", engineered_features.columns.tolist())
print("Engineered input types:", engineered_features.dtypes.value_counts().to_dict())
print("Missing entries in engineered inputs:", int(engineered_features.isna().sum().sum()))
print("All-engineered-inputs numeric?", all(pd.api.types.is_numeric_dtype(t) for t in engineered_features.dtypes))
print("Target excluded?", "species" not in engineered_features.columns)
print("Split role excluded?", "split_role" not in engineered_features.columns)

### What did we gain, and what may be redundant?

| Change | What it provides | What to watch |
|:--|:--|:--|
| Converted four measurements and the ratio to z-scores | Comparable reference scales | Original units are less visible; parameters must be retained |
| Added bill-shape ratio | Explicit bill proportion | Depends on existing bill measurements; may be redundant |
| Replaced one island label with three indicators | Numeric membership without artificial ranking | More columns; location may be an unreliable shortcut |
| Excluded two rows missing a needed measurement | Complete numerical inputs for this example | Some observations are no longer represented |
| Kept `species` in a separate Series | Clear input–target separation | Must preserve row alignment when later splitting or training |

**More features ≠ better features.** We have not used a learning algorithm, so we cannot measure whether this representation improves any prediction.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">08 · PREPARING FEATURES RESPONSIBLY</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">What must we learn from training data only?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">The same idea from L07 applies to scales, categories, and data-dependent boundaries.</div>
</div>

### Two kinds of transformation decisions

**Fixed, domain-based transformations** do not need to look across the dataset to learn parameters:

- Grams → kilograms (divide by 1,000).
- Bill length ÷ bill depth (a known mathematical relationship).
- A verified mapping of labels, or bins with **externally specified** thresholds.

**Data-dependent transformations** estimate something from observations:

- Min/max values used by min–max scaling.
- Mean and SD used by standardization.
- Quantiles chosen as bin boundaries.
- Category vocabulary learned from the observed training subset.

**Rule for later ML workflows:** First assign training/validation/test roles. Fit the data-dependent parameters on **training data only**, then apply the saved parameters unchanged to validation, test, and future observations.

<div style="background:#102A43;color:#ffffff;border-left:7px solid #18B7AC;padding:15px 20px;border-radius:7px;margin-top:26px;margin-bottom:10px;">
<div style="color:#63DDD3;font-size:12px;letter-spacing:1.3px;font-weight:700;">09 · SYNTHESIS AND NEXT STEPS</div>
<div style="font-size:23px;font-weight:700;line-height:1.4;">What makes a useful feature representation?</div>
<div style="font-size:14px;color:#DDEAF3;margin-top:4px;">Review the decisions, then carry the challenge into the weekly assignment.</div>
</div>

### Our feature-engineering decision checklist

| Ask yourself | Why the question matters |
|:--|:--|
| **What is my target, and what inputs are available?** | Prevents inappropriate features and target leakage. |
| **What does this measurement or label mean?** | Prevents arbitrary transformations with no justification. |
| **Can a derived feature express an important relationship?** | A ratio or difference may reveal a new view of existing evidence. |
| **What details will the transformation preserve or lose?** | Binning and compression can change distinctions between observations. |
| **Do different units create misleading numerical magnitudes?** | Scale-sensitive methods may treat features unequally. |
| **Does a category have a meaningful order?** | Arbitrary integer labels can invent false numerical relations. |
| **Where did the transformation parameters come from?** | Training-only fitting protects the integrity of evaluation. |
| **Have I inspected the final table?** | Row count, numeric types, missingness, column names, and roles must be verified. |

**Central conclusion:** Good feature engineering is about **representing useful information meaningfully**, not generating the largest possible number of features.

## Assignment 04 — Preparing Your Data for Machine Learning

### The Challenge

In **Assignment 03**, you divided the Palmer Penguins dataset into training (70%), validation (20%), and test (10%) sets.

Now, your task is to **prepare these same observations for Machine Learning**, following the feature-engineering steps demonstrated in **Section 7 of Lecture 08**.

Start with your own `penguins_with_splits.csv` file from Assignment 03. **Keep the original split assignments unchanged.**

### Your Tasks

1. **Load and inspect your dataset.**
   - Verify the `train`, `val`, and `test` labels.
   - Check for missing values and decide how to handle them.
   - Preserve the original dataset.

2. **Prepare your features.**
   - Separate `species` (the target) from the input features.
   - Use the physical measurements and `island`, as demonstrated in Section 7.
   - Create the `bill_shape_ratio` feature.

3. **Transform your features.**
   - Standardize the numerical features using the **training set's mean and standard deviation**.
   - Apply those same training statistics to the validation and test sets.
   - Apply one-hot encoding to `island`, using categories determined from the training set. If another split contains an unseen category, explain how you would handle it rather than learning new categories from that split.

4. **Inspect your final dataset.**
   - Display the resulting feature table.
   - Check its dimensions, data types, and missing values.
   - Verify that `species` and `split` are not included as input features.
   - Compare the original and transformed representations.

### Important Rules

- **Avoid data leakage:** Never calculate scaling parameters or learn encoding categories using validation or test data.
- Keep the original training, validation, and test assignments.
- Keep the target separately from the input features.

### Submission

Submit **`Assignment04.ipynb`**, containing:

- Your complete Python implementation.
- The final transformed feature table, preserving the original split assignments separately.
- A few short explanations of your preprocessing decisions.


### Looking ahead — Lecture 09: Measuring Similarity

We have progressed from **reliable observations** to **meaningful numerical representations**:

- **L07:** Can we trust and clean our observations?
- **L08:** Which information should we represent, and how should we transform it?
- **L09:** How do we measure whether two observations are similar?
- **L10:** Which features should we select, and how can we organize transformations into reliable pipelines?

**Our next question:**

> **We now have numerical representations of our penguins. But how can we decide whether two penguins are similar—and how does the way we represent their features influence that comparison?**

That is the starting point of **L09 — Measuring Similarity: Distances and Proximity Measures**.

### Sources, acknowledgements, and further reading

- **Palmer Penguins:** Horst, A. M., Hill, A. P., & Gorman, K. B. (2020), *palmerpenguins: Palmer Archipelago (Antarctica) penguin data* ([project](https://allisonhorst.github.io/palmerpenguins/); [Zenodo DOI 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)). Ecological observations originate from Gorman, K. B., Williams, T. D., & Fraser, W. R. (2014), *PLoS ONE*. This notebook uses the course's bundled 344-row CSV.
- **Teaching inspirations:** Selected concepts were adapted from the teaching material of **Prof. Antonio Liotta**.
- **Pandas references:** [`pd.cut`](https://pandas.pydata.org/docs/reference/api/pandas.cut.html), [`pd.qcut`](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html), [`pd.get_dummies`](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html), [`Series.std`](https://pandas.pydata.org/docs/reference/api/pandas.Series.std.html).
- **NumPy references:** [`np.log10`](https://numpy.org/doc/stable/reference/generated/numpy.log10.html), [`np.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html).

The dataset, code, and visualizations are included locally; running this notebook does not require an internet connection.